# Hands-on: Train a CNN to Recognize Bean-Leaf Disease

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lobaton/AI-Workshop-UNTRM/blob/main/session2_inside_the_black_box/src/CNN_colab.ipynb)

This notebook lets you **run the bean-leaf CNN yourself**. The ideas behind it (filters, feature maps,
what the layers learn) are explained in the presenter's [marimo notebook](https://molab.marimo.io/github/lobaton/AI-Workshop-UNTRM/blob/main/session2_inside_the_black_box/src/CNN.py).

**No coding needed:** run each cell in order with the ▶ button (or `Shift + Enter`). Cells marked
**✏️ Settings** have a few values you can change before running them.

**Tip:** for faster training, choose *Runtime → Change runtime type → T4 GPU*.

## 1. Setup
Downloads the shared workshop code and the bean-leaf images (128×128 pixels).

In [ ]:
import os, urllib.request
if not os.path.exists("cnn_common.py"):
    urllib.request.urlretrieve(
        "https://raw.githubusercontent.com/lobaton/AI-Workshop-UNTRM/main/session2_inside_the_black_box/src/cnn_common.py", "cnn_common.py")

import numpy as np
import matplotlib.pyplot as plt
import torch
from sklearn.metrics import ConfusionMatrixDisplay
import cnn_common as cc

plt.rcParams.update({"font.size": 14, "lines.linewidth": 3})
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using", "GPU" if device == "cuda" else "CPU (training will be slower)")

data = cc.load_beans(cc.fetch(cc.BEANS_FILE))
for split, (x, y) in data.items():
    print(f"{split:>10}: {len(y)} images")

## 2. ✏️ Settings

Change these values and re-run the cells below to compare results:

- `N_FILTERS`: number of filters in the first layer (later layers use 2× and 4× as many).
- `EPOCHS`: number of passes through the training images.
- `N_TRAIN`: how many training images to use (at most 1034).
- `USE_AUGMENT`: if `True`, each training image is randomly flipped or rotated, so the network sees
  more variety (a leaf is still the same leaf when turned around).

In [ ]:
N_FILTERS = 16
EPOCHS = 20
N_TRAIN = 1034
USE_AUGMENT = True

## 3. Train

In [ ]:
torch.manual_seed(0)
model = cc.BeansCNN(N_FILTERS)
print(f"The network has {cc.count_params(model):,} parameters")

x_tr, y_tr = cc.random_subset(*data["train"], N_TRAIN)
x_val, y_val = data["validation"]
history = cc.train_model(
    model, cc.to_tensor(x_tr), torch.from_numpy(y_tr), cc.to_tensor(x_val), torch.from_numpy(y_val),
    epochs=EPOCHS, use_augment=USE_AUGMENT, device=device, log=print)
print(f"Training took {history['seconds']:.0f} seconds")

## 4. Results

The learning curves show how the training and validation accuracy change over the epochs. The **test
accuracy** is measured once, on images the network has never seen. Chance level (guessing) is 33%.

In [ ]:
x_test, y_test = data["test"]
_, test_acc, test_preds = cc.evaluate(model, cc.to_tensor(x_test), torch.from_numpy(y_test))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
epochs = range(1, len(history["train_acc"]) + 1)
axes[0].plot(epochs, 100 * np.array(history["train_acc"]), "o-", label="Training")
axes[0].plot(epochs, 100 * np.array(history["val_acc"]), "s--", label="Validation")
axes[0].axhline(100 / 3, color="gray", linestyle=":", label="Chance")
axes[0].set(xlabel="Epoch", ylabel="Accuracy (%)", ylim=(0, 100), title="Learning curve")
axes[0].legend()
ConfusionMatrixDisplay.from_predictions(
    y_test, test_preds, display_labels=[n.replace(" ", "\n", 1) for n in cc.CLASS_NAMES],
    cmap="Blues", colorbar=False, ax=axes[1])
axes[1].set_title(f"Test accuracy: {test_acc:.0%}")
plt.tight_layout()
plt.show()

**Try:** turn `USE_AUGMENT` off, or use only 200 training images. How does the gap between the
training and validation curves change?

## 5. Try your own leaf

Upload a photo of a leaf (for example, taken with your phone). The photo is cropped to a square, resized
to 128×128 and passed through the network you just trained. If you are not in Colab, or skip the upload,
a test image is used instead.

Think about the result: the network was trained only on **bean** leaves photographed in **Uganda**. What
does it answer for a coffee leaf, or for a photo with a different background?

In [ ]:
from PIL import Image

try:
    from google.colab import files
    uploaded = files.upload()
    photo = Image.open(next(iter(uploaded))).convert("RGB") if uploaded else None
except ImportError:
    photo = None

if photo is None:
    photo = Image.fromarray(x_test[np.flatnonzero(y_test == 1)[0]])
    print("No photo uploaded: using a bean rust image from the test set.")

side = min(photo.size)
left, top = (photo.width - side) // 2, (photo.height - side) // 2
leaf = np.asarray(photo.crop((left, top, left + side, top + side)).resize((128, 128)))

model.eval()
with torch.no_grad():
    probs = torch.softmax(model(cc.to_tensor(leaf[None]).to(device)), dim=1)[0].cpu().numpy()

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].imshow(leaf)
axes[0].axis("off")
axes[1].barh(cc.CLASS_NAMES, 100 * probs, color=["#E69F00", "#D55E00", "#009E73"])
axes[1].set(xlim=(0, 100), xlabel="Probability (%)", title="Network's answer")
plt.tight_layout()
plt.show()

## Questions for discussion

1. The network gives an answer even for a photo that is not a bean leaf at all. Why? What would you need
   to change so that it can say "I don't know"?
2. If you wanted to use this idea for coffee leaf rust in Amazonas, what data would you need to collect?